# 09 · Executive Summary & Report Generation

The management-facing wrap-up. We load the persisted results and render a self-contained HTML report (`reports/html/index.html`) that leads with dollars, not ROC-AUC.

In [1]:
# project-root bootstrap
import os
from pathlib import Path
_p=Path.cwd()
while not (_p/'config'/'config.yaml').exists() and _p!=_p.parent:
    _p=_p.parent
os.chdir(_p); print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


In [2]:
import json
from fraud.report import render_report

with open('reports/results.json') as f:
    R = json.load(f)
R

{'net_saved': 2496.73,
 'baseline_loss': 4003.06,
 'fraud_caught_pct': 0.6397,
 'threshold': 0.1875,
 'recall': 0.898,
 'precision': 0.7333,
 'pr_auc': 0.8855,
 'roc_auc': 0.9774,
 'precision_at_k': 0.45,
 'k': 100,
 'shap_top': ['V14', 'V4', 'V12', 'V10', 'V11', 'V3', 'V7', 'V16'],
 'figures': ['../figures/threshold_sweep.png',
  '../figures/confusion_matrices.png',
  '../figures/diagnostics.png',
  '../figures/robustness_segments.png',
  '../figures/shap_summary.png']}

## The story in three numbers

In [3]:
print('EXECUTIVE SUMMARY — credit-card fraud detection (held-out test set)')
print('=' * 64)
print(f"  Net dollars saved vs no model : ${R['net_saved']:,.0f}")
print(f"  Fraud DOLLARS recovered       : {R['fraud_caught_pct']*100:.1f}%")
print(f"  Fraud CASES caught (recall)   : {R['recall']*100:.1f}%")
print('-' * 64)
gap = R['recall']*100 - R['fraud_caught_pct']*100
print(f"  Insight: we catch {R['recall']*100:.0f}% of fraud CASES but only "
      f"{R['fraud_caught_pct']*100:.0f}% of fraud DOLLARS —")
print(f"  the missed cases skew to high-value fraud (a {gap:.0f}pt gap). "
      "Priority: improve recall on large amounts.")
print('-' * 64)
print(f"  Decision threshold            : {R['threshold']}")
print(f"  Precision @ k={R['k']} alerts      : {R['precision_at_k']}")
print(f"  Top fraud drivers (SHAP)      : {', '.join(R['shap_top'][:5])}")

EXECUTIVE SUMMARY — credit-card fraud detection (held-out test set)
  Net dollars saved vs no model : $2,497
  Fraud DOLLARS recovered       : 64.0%
  Fraud CASES caught (recall)   : 89.8%
----------------------------------------------------------------
  Insight: we catch 90% of fraud CASES but only 64% of fraud DOLLARS —
  the missed cases skew to high-value fraud (a 26pt gap). Priority: improve recall on large amounts.
----------------------------------------------------------------
  Decision threshold            : 0.1875
  Precision @ k=100 alerts      : 0.45
  Top fraud drivers (SHAP)      : V14, V4, V12, V10, V11


## Render the HTML report

In [4]:
render_report(R, 'reports/html/index.html')
print('wrote reports/html/index.html')
from IPython.display import IFrame
IFrame('../reports/html/index.html', width='100%', height=480)

wrote reports/html/index.html


**Deliverable:** open `reports/html/index.html` in a browser for the management view. This closes the pipeline: ingestion → quality → preprocessing → features → model → cost decision → executive report.